In [21]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pickle # lưu model sau khi training

# Import tf-idf và cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Import success!")

Import success!


I. Xây dựng thuật toán


In [22]:
def load_data():
    load_dotenv()
    try:
        connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
        engine = create_engine(connection_string)
        print("Kết nối CSDL thành công!")
    except Exception as e:
        print(f"LỖI kết nối: {e}")
        return None, None

    # Lấy dữ liệu phim với các thông tin: genre, actor, director, summary để làm Item Profile
    sql_items = """
        SELECT 
            m.id, m.title, m.summary AS description,
            COALESCE(g.genres, '') AS genres,
            COALESCE(a.actors, '') AS actors,
            COALESCE(d.directors, '') AS directors
        FROM Movies m
        LEFT JOIN (
            SELECT movie_id, STRING_AGG(G.name, ', ') AS genres 
            FROM Movie_Genres MG JOIN Genres G ON MG.genre_id = G.id GROUP BY movie_id
        ) g ON m.id = g.movie_id
        LEFT JOIN (
            SELECT movie_id, STRING_AGG(A.name, ', ') AS actors 
            FROM Movie_Actors MA JOIN Actors A ON MA.actor_id = A.id GROUP BY movie_id
        ) a ON m.id = a.movie_id
        LEFT JOIN (
            SELECT movie_id, STRING_AGG(D.name, ', ') AS directors 
            FROM Movie_Directors MD JOIN Directors D ON MD.director_id = D.id GROUP BY movie_id
        ) d ON m.id = d.movie_id
    """

    # Lấy dữ liệu tương tác để làm User Profile
    sql_interactions = """
        SELECT user_id, movie_id, base_score, last_watched_at 
        FROM view_movie_recommendation_scoring
    """

    try:
        print("Đang tải dữ liệu từ Postgres...")
        df_items = pd.read_sql(sql_items, engine)
        df_interactions = pd.read_sql(sql_interactions, engine)
        
        # Đảm bảo cột thời gian đúng định dạng để tính heuristic_score sau này
        df_interactions['last_watched_at'] = pd.to_datetime(df_interactions['last_watched_at'])
        
        print(f"Tải thành công: {len(df_items)} phim và {len(df_interactions)} tương tác.")
        return df_items, df_interactions

    except Exception as e:
        print(f"LỖI tải dữ liệu: {e}")
        return None, None

In [23]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            # Xóa khoảng trắng giữa các tên riêng (ví dụ: 'Action, Sci-Fi' -> 'action,sci-fi')
            # Sau đó xóa dấu phẩy để tạo thành chuỗi các từ đơn
            return x.replace(" ", "").lower().replace(",", " ")
        else:
            return ''

In [24]:
def create_soup(data_row):
    # Làm sạch các trường dữ liệu
    genres = clean_data(data_row['genres'])
    actors = clean_data(data_row['actors'])
    directors = clean_data(data_row['directors'])
    description = data_row['description'].lower() if data_row['description'] else ""

    
    return f"{genres} {actors} {directors} {description}"

In [25]:
df_items, df_interactions = load_data()
df_items.head()

Kết nối CSDL thành công!
Đang tải dữ liệu từ Postgres...
Tải thành công: 1554 phim và 12 tương tác.


,id,title,description,genres,actors,directors
0,11,Star Wars,Princess Leia is captured and held hostage by ...,"Adventure, Action, Science Fiction","Mark Hamill, Alec Guinness, Peter Cushing, Car...",George Lucas
1,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Animation, Family","Geoffrey Rush, Willem Dafoe, Alexander Gould, ...",Andrew Stanton
2,13,Forrest Gump,A man with a low IQ has accomplished great thi...,"Drama, Comedy, Romance","Gary Sinise, Mykelti Williamson, Tom Hanks, Ro...",Robert Zemeckis
3,18,The Fifth Element,"In 2257, a taxi driver is unintentionally give...","Adventure, Action, Science Fiction","Chris Tucker, Ian Holm, Gary Oldman, Milla Jov...",Luc Besson
4,22,Pirates of the Caribbean: The Curse of the Bla...,After Port Royal is attacked and pillaged by a...,"Adventure, Fantasy, Action","Keira Knightley, Jack Davenport, Orlando Bloom...",Gore Verbinski


In [26]:
df_interactions.head()

,user_id,movie_id,base_score,last_watched_at
0,3,278,5.836242,2026-03-09 22:38:28.348478
1,3,1084242,3.226846,2026-03-09 22:38:28.348478
2,7,1003596,7.671141,2026-03-09 22:38:28.348478
3,7,299536,7.425503,2026-03-09 22:38:28.348478
4,7,315635,7.083221,2026-03-09 22:38:28.348478


In [27]:
# Tạo cột 'soup' bằng cách kết hợp các trường metadata 
df_items['soup'] = df_items.apply(create_soup, axis=1)
df_items.head()

,id,title,description,genres,actors,directors,soup
0,11,Star Wars,Princess Leia is captured and held hostage by ...,"Adventure, Action, Science Fiction","Mark Hamill, Alec Guinness, Peter Cushing, Car...",George Lucas,adventure action sciencefiction markhamill ale...
1,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Animation, Family","Geoffrey Rush, Willem Dafoe, Alexander Gould, ...",Andrew Stanton,animation family geoffreyrush willemdafoe alex...
2,13,Forrest Gump,A man with a low IQ has accomplished great thi...,"Drama, Comedy, Romance","Gary Sinise, Mykelti Williamson, Tom Hanks, Ro...",Robert Zemeckis,drama comedy romance garysinise mykeltiwilliam...
3,18,The Fifth Element,"In 2257, a taxi driver is unintentionally give...","Adventure, Action, Science Fiction","Chris Tucker, Ian Holm, Gary Oldman, Milla Jov...",Luc Besson,adventure action sciencefiction christucker ia...
4,22,Pirates of the Caribbean: The Curse of the Bla...,After Port Royal is attacked and pillaged by a...,"Adventure, Fantasy, Action","Keira Knightley, Jack Davenport, Orlando Bloom...",Gore Verbinski,adventure fantasy action keiraknightley jackda...


In [28]:
# Tính TF-IDF matrix cho cột 'soup'
vectorize = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorize.fit_transform(df_items['soup'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (1554, 17910)


In [29]:
#import SVD để giảm chiều dữ liệu
from sklearn.decomposition import TruncatedSVD

In [30]:
svd = TruncatedSVD(n_components=100, random_state=42)

latent_matrix = svd.fit_transform(tfidf_matrix)

print("Latent matrix shape after SVD:", latent_matrix.shape)

Latent matrix shape after SVD: (1554, 100)


In [55]:
# Lưu bảng tra cứu id -> index
indices = pd.Series(df_items.index, index=df_items['id']).drop_duplicates()
indices[299536]

np.int64(715)

In [36]:
half_life_days = 30

lambda_val = np.log(2) / half_life_days

print("Lambda value for time decay:", lambda_val)

Lambda value for time decay: 0.023104906018664842


In [81]:
def build_user_profile(user_id, df_interactions, latent_matrix, indices, lambda_param):
    # Lọc tương tác của user
    user_interactions = df_interactions[df_interactions['user_id'] == user_id].copy()

    
    if user_interactions.empty:
        return None  # Trả về None nếu user chưa có tương tác nào
    
    # Lấy thời điểm mới nhất mà user xem để làm mốc dữ liệu
    ref_time = user_interactions['last_watched_at'].max()
    days_diff = (ref_time - user_interactions['last_watched_at']).dt.days


    # Tính điểm heuristic cho mỗi tương tác
    user_interactions['heuristic_score'] = user_interactions['base_score'] * np.exp(-lambda_param * days_diff)

    valid_movie_ids = set(indices.keys())

    # Lấy latent vectors của các phim đã xem 
    valid_interactions = user_interactions[user_interactions['movie_id'].isin(valid_movie_ids)]

    if valid_interactions.empty:
        return None  # Trả về None nếu không có phim nào trong latent matrix
    
    # Tính user profile
    movie_idxs = [indices[movie_id] for movie_id in valid_interactions['movie_id']]
    weights = valid_interactions['heuristic_score'].values.reshape(-1, 1)

    movie_vectors = latent_matrix[movie_idxs]
    user_profile = np.sum(movie_vectors * weights, axis=0) / np.sum(weights)

    return user_profile, valid_interactions

In [82]:
user_id = 7
user_profile, valid_interactions = build_user_profile(user_id, df_interactions, latent_matrix, indices, lambda_val)

print("User profile vector shape:", user_profile)

User profile vector shape: [ 0.11051363  0.01616351  0.08281524  0.00690514  0.03907372 -0.07106949
 -0.02005883  0.01571233  0.0141568  -0.03248283  0.03659463  0.03577622
 -0.01397524  0.01361774 -0.01587205 -0.03115583  0.05018838 -0.0067747
  0.00209358  0.01873963 -0.01444414  0.00659511 -0.01508995  0.02264063
  0.02051706  0.00688765  0.0104371  -0.00327168 -0.02443325  0.01194784
 -0.01805199 -0.02988671  0.04352906  0.00327329 -0.02626673 -0.03135855
  0.01093319 -0.00600329 -0.01709013 -0.05184683 -0.02342827 -0.03963633
 -0.00694431  0.03240854  0.03626554 -0.0138403   0.00988395  0.00459527
 -0.02622229 -0.02160684 -0.01432105 -0.04623892 -0.06221244 -0.04771611
  0.01837468  0.01654958 -0.0184512   0.01942033 -0.02053982  0.00569113
 -0.02132587  0.00618149 -0.00335386  0.01567486 -0.01101346  0.01504965
 -0.00765565 -0.01224031 -0.02836386  0.05217615  0.03176529  0.00848187
 -0.01814978 -0.03022177  0.01363043 -0.02712884 -0.0085938  -0.00387047
  0.0385985  -0.02262534 

In [83]:
valid_interactions

,user_id,movie_id,base_score,last_watched_at,heuristic_score
2,7,1003596,7.671141,2026-03-09 22:38:28.348478,7.671141
3,7,299536,7.425503,2026-03-09 22:38:28.348478,7.425503
4,7,315635,7.083221,2026-03-09 22:38:28.348478,7.083221
5,7,1228246,5.718121,2026-03-09 22:38:28.348478,5.718121
8,7,1377505,5.362416,2026-03-09 22:38:28.348478,5.362416
10,7,507089,5.744966,2026-03-09 22:38:28.348478,5.744966


In [88]:
#Lấy ra danh sách các movie_id đã xem
seen_movie_ids = valid_interactions['movie_id'].unique()
seen_movie_ids

array([1003596,  299536,  315635, 1228246, 1377505,  507089])

In [89]:
# Chuyển các movie_id đã xem sang index trong indices
seen_movie_indices = [indices[movie_id] for movie_id in seen_movie_ids if movie_id in indices]
seen_movie_indices

[np.int64(1150),
 np.int64(715),
 np.int64(725),
 np.int64(1326),
 np.int64(1465),
 np.int64(872)]

In [92]:
similarities = cosine_similarity(user_profile.reshape(1, -1), latent_matrix).flatten()

similarities[seen_movie_indices] = -1

top_10_indices = similarities.argsort()[-30:][::-1]

In [93]:
recommendations = df_items.iloc[top_10_indices][['id', 'title', 'genres']]
print(f"--- Top 30 gợi ý cho User {user_id} ---")
print(recommendations)

--- Top 30 gợi ý cho User 7 ---
           id                                        title  \
689    271110                   Captain America: Civil War   
714    299534                            Avengers: Endgame   
543     99861                      Avengers: Age of Ultron   
225     10138                                   Iron Man 2   
138      1726                                     Iron Man   
326     24428                                 The Avengers   
352     31385                             Zombie Nightmare   
473     68721                                   Iron Man 3   
967    634649                      Spider-Man: No Way Home   
68        557                                   Spider-Man   
1060   823464              Godzilla x Kong: The New Empire   
496     76341                           Mad Max: Fury Road   
694    284053                               Thor: Ragnarok   
169      5421                                       Absurd   
1418  1309012                         

II. Sử dụng Precision@K, Recall@K, và F1-score để kiểm tra.
- Nguyên tắc: 1 phim được coi là liên quan đến user nếu số các thể loại user thích trong phim đó > số thể loại user không thích (Comparision of selected algorithms in Movie Recommender System)
- Cách tính: 
+ Precision@K: 